# Oefening recommender systems

Installatie package voor deze les: `conda install -c conda-forge implicit` OF `pip install implicit`. Mogelijk zijn er nog packages nodig zoals `pip install h5py`

We vertrekken van simpele voorbeelddata:

In [1]:
import pandas as pd

data = {
    'user_id': [1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6],
    'song_id': [1, 2, 1, 3, 4, 5, 5, 3, 5, 2, 4, 6],
    'rating': [4, 3, 5, 2, 4, 1, 3, 5, 2, 4, 3, 5]
}

df = pd.DataFrame(data)


Zet de matrix met behulp van een df.pivot_table om naar een werkbaar formaat (rijen: users; kolommen: films)

In [2]:
user_item_matrix = df.pivot_table(index='user_id', columns='song_id', values='rating', fill_value=0)

In [3]:
user_item_matrix.head()

song_id,1,2,3,4,5,6
user_id,,,,,,
1,4,3,0,0,0,0
2,5,0,2,0,0,0
3,0,0,0,4,2,0
4,0,0,5,0,2,0
5,0,4,0,3,0,0


In [4]:

from scipy.sparse import csr_matrix
sparse_user_item_matrix = csr_matrix(user_item_matrix.values)

In [5]:
import implicit

# initialize a model
model = implicit.als.AlternatingLeastSquares(factors=50)

# train the model on a sparse matrix of user/item/confidence weights
model.fit(sparse_user_item_matrix)

user_id = 1
# recommend items for a user
recommendations = model.recommend(user_id, sparse_user_item_matrix[user_id])
print(recommendations)

item_id = 1
# find related items
related = model.similar_items(item_id)
print(related)

/home/bruno/miniconda3/envs/ml_algorithms/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/bruno/miniconda3/envs/ml_algorithms/lib/python3.11/site-packages/implicit/cpu/als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 6 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()
100%|██████████| 15/15 [00:01<00:00, 12.05it/s]

(array([1, 4, 5, 3, 2, 0, 0, 0, 0, 0], dtype=int32), array([ 4.1330829e-03,  2.9625595e-03,  1.6018748e-06, -5.8588758e-03,
       -3.4028235e+38, -3.4028235e+38,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00], dtype=float32))
(array([1, 0, 5, 4, 3, 2, 0, 0, 0, 0], dtype=int32), array([ 9.9999988e-01,  6.9394961e-02, -1.1045907e-04, -3.3885441e-04,
       -6.3815424e-03, -1.4758654e-01,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00], dtype=float32))


# uitgewerkt voorbeeld - Muziek aanbevelen met last.fm 360K dataset.

### Dataset binnenhalen
Eerst data binnenhalen:

In [6]:
from implicit.datasets.lastfm import get_lastfm
artists, users, artist_user_plays = get_lastfm()

184MB [00:07, 25.8MB/s]                              


`artist_user_plays` is een ijle matrix, waarbij elke rij overeenkomt met een andere muzikant en elke kolom overeenkomt met een andere gebruiker. De niet-nulle entries in de `artist_user_plays`-matrix bevatten het aantal keren dat de gebruiker die artiest heeft afgespeeld. De variabelen `artists` en `users` zijn arrays die labels bevatten voor elke rij en kolom in de `artist_user_plays`-matrix.

De `implicit` package richt zich uitsluitend op aanbevelingssystemen voor impliciete feedback, waarbij we positieve voorbeelden krijgen van wat de gebruiker heeft geïnteracteerd, maar niet de bijbehorende negatieve voorbeelden van waar gebruikers niet in geïnteresseerd zijn. In dit voorbeeld worden we het aantal keren getoond dat de gebruiker een artiest in de dataset heeft afgespeeld, en kunnen we afleiden dat een hoog afspeelaantal aangeeft dat de gebruiker een artiest leuk vindt. We kunnen echter niet afleiden dat alleen omdat de gebruiker een band nog nooit heeft afgespeeld, dit betekent dat de gebruiker de band niet leuk vindt.

## Het model trainen

Implicit biedt implementaties van verschillende algoritmen voor impliciete recommender systemen. Voor dit voorbeeld kijken we naar het 'AlternatingLeastSquares'-model. Dit model is bedoeld om een binair target te leren of elke gebruiker interactie heeft gehad met elk item - maar weegt elke binaire interactie met een betrouwbaarheidswaarde van hoe zeker we zijn van deze interactie tussen gebruiker en item. De impliciete implementatie maakt gebruik van de waarden van een ijle matrix om de betrouwbaarheid weer te geven, waarbij de niet-nul waardes aangeven of de gebruiker al dan niet interactie heeft gehad met het item.

In [8]:
from implicit.nearest_neighbours import bm25_weight

# weeg de matrix, om zo sommige users die steeds hetzelfde luisteren in te perken
# en om bepaalde héél populaire liedjes te herschalen
artist_user_plays = bm25_weight(artist_user_plays, K1=100, B=0.8)

# we hebben straks (user, item) sparse matrices nodig in plaats van (item, user)
user_plays = artist_user_plays.T.tocsr()

Nu kunnen we trainen, dit neemt wel wat tijd in beslag:

In [9]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0)
model.fit(user_plays)

100%|██████████| 15/15 [01:38<00:00,  6.58s/it]


### Voorspellingen

Nu kunnen er voorspellingen worden gemaakt. Dit kan zowel individueel als in batch.

In [16]:
# Voorspelling voor 1 user:
userid = 12345
ids, scores = model.recommend(userid, user_plays[userid], N=10, filter_already_liked_items=True)

De `.recommend`-functie zal de *N* beste aanbevelingen geven voor elke user in de input, en de itemids teruggeven in de `ids` array, (en ook de berekende scores in de `score` array). Je kunt dus inschatten hoe groot het verschil tussen items was. De muzikanten die worden aanbevolen voor elke user zitten in de ids van de `artist` array:

In [29]:
type(artists)

numpy.ndarray

In [18]:
# resultaat in een tabel zetten
import numpy as np
import pandas as pd
pd.DataFrame({"artist": artists[ids], "score": scores, "already_liked": np.isin(ids, user_plays[userid].indices)})

,artist,score,already_liked
0,laibach,1.042190,False
1,ordo equilibrio,0.974190,False
2,gåte,0.973895,False
3,ordo rosarius equilibrio,0.963739,False
4,of the wand and the moon,0.960298,False
5,blood axis,0.957923,False
6,shining,0.957231,False
7,ice ages,0.942891,False
8,khold,0.939586,False
9,karjalan sissit,0.935883,False


De `already_liked` toont of een user al heeft geinterageerd met het item, en in dit voorbeeld is dit voor de meeste items het geval. Je kunt deze reuit filteren met de parameter `filter_already_liked_items`. De `user_plays [userid]`
 parameter is gebruikt om op te zoeken met welke items een user heeft geïnterageerd, je kan dit op `None` zetten als je de voorkeuren van de eigen user niet wil meenemen.

 Er zijn ook opties in de `filter_items` parameter en de `items` parameter, je kan de voorstelling van de user ook on the fly herberekenen met `recalculate_user` parameter.

Meer details kan je online vinden in de documentatie van deze library.

## Gelijkaardige items tonen

Volgende code toont items gelijkaardig aan *The Beatles* (zoek op 'the beatles', alles met kleine letters):

In [45]:
id_other_artist = np.where(artists=='metallica')[0][0]
print(id_other_artist)

186258


In [46]:
# get related items for the beatles (itemid = 252512)
ids, scores= model.similar_items(id_other_artist)

# display the results using pandas for nicer formatting
pd.DataFrame({"artist": artists[ids], "score": scores})

,artist,score
0,metallica,1.000000
1,iron maiden,0.994843
2,ac/dc,0.993514
3,guns n' roses,0.993239
4,guns n roses,0.992443
5,system of a down,0.992385
6,rammstein,0.991546
7,megadeth,0.991271
8,ozzy osbourne,0.991094
9,pantera,0.990681


### Batch recommendations

De `.recommend`, `.similar_items` en `.similar_users` functies hebben allemaal de mogelijkheid om batches te berekenen. Je kunt een array van userids of itemids meegeven, and dan geeft het systeem een 2D array terug van ids en scores, met in elke rij van de output matrix corresponderend met de waarde van de input. Dit is vooral veel efficiënter, omdat dit in parallel kan worden berekend op grote batches (véél beter dan for loops gebruiken).

In [47]:
userids = np.arange(1000)
ids, scores = model.recommend(userids, user_plays[userids])
ids, ids.shape

(array([[161850, 107119, 165938, ...,  76757, 136336, 205631],
        [128505,  71465, 252956, ..., 262990,  71225, 267949],
        [186835, 238422, 120981, ..., 167270, 158697, 131061],
        ...,
        [ 83885, 265625, 151783, ..., 268237, 202346, 149104],
        [109930,  97970, 116857, ..., 236697, 146064, 237577],
        [ 21090, 158209,  69978, ..., 158452, 185387, 272293]], dtype=int32),
 (1000, 10))